# Agent1: base-model grounding benchmark
Run locally in VS Code using the Agent1 `.venv` kernel. This notebook evaluates the original Llama model only; no LoRA adapter is loaded.

**Before starting:** shut down the kernel of notebook 08 to release its model from GPU memory. Closing its tab alone may not stop that kernel. Place `agent1_grounding_benchmark_v1.json` in `data/evaluation/`.

Run the cells in order. Generated responses are saved after every case. No training or automatic correctness grading occurs here. Keep this diagnostic set out of training; an independent final test will be needed later.


In [1]:
from pathlib import Path
import json
import hashlib
import importlib.metadata as metadata
from datetime import datetime, timezone
from time import perf_counter

from agent1.generation.response_prompt import build_response_messages

project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "src" / "agent1").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the Agent1 project root or a subfolder.")

benchmark_path = project_root / "data" / "evaluation" / "agent1_grounding_benchmark_v1.json"
benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
cases = benchmark["cases"]
assert len(cases) == 24 and len({case["id"] for case in cases}) == 24

# Freeze the exact input messages for this run. Evaluation metadata stays out.
prompts = {
    case["id"]: build_response_messages(
        user_message=case["user_message"],
        knowledge_context=case["knowledge_context"],
        tool_result=case["tool_result"],
    )
    for case in cases
}
print(f"Loaded {len(cases)} benchmark cases.")


Loaded 24 benchmark cases.


## Load the original base model once
Use the same base model, NF4 settings, and GPU placement as the working local setup. The original base tokenizer is used, without loading adapter weights. Cached model files should be reused.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "CUDA-enabled PyTorch is required."
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
MAX_NEW_TOKENS = 128  # Keep this identical for v2 evaluation.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model.eval()
model.config.use_cache = True
print("Loaded base model only on:", torch.cuda.get_device_name(0))


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

W1024 11:51:44.100000 16468 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loaded base model only on: NVIDIA GeForce GTX 1050


In [3]:
def generate_case(messages):
    torch.cuda.synchronize()
    started = perf_counter()
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    torch.cuda.synchronize()
    elapsed = perf_counter() - started
    new_ids = output[0, inputs["input_ids"].shape[1]:].tolist()

    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id
    eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])
    ended_with_eos = bool(new_ids) and new_ids[-1] in eos_ids

    return {
        "answer": tokenizer.decode(
            new_ids, skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip(),
        "input_tokens": inputs["input_ids"].shape[1],
        "generated_tokens": len(new_ids),
        "generation_seconds": round(elapsed, 3),
        "ended_with_eos": ended_with_eos,
        "hit_token_limit": len(new_ids) >= MAX_NEW_TOKENS and not ended_with_eos,
    }


## Create a results folder
Run this setup cell once per experiment. Each run gets a unique folder under `experiments/grounding/`. The saved manifest includes benchmark content, exact model inputs, tokenizer template, package versions, and generation settings. Use these same inputs/settings for v2.

If the next cell is interrupted, rerun **only the next cell** to skip cases already saved. After a kernel restart, reload the model and set `run_dir` to the printed existing folder instead of creating a new run.


In [4]:
run_id = "base_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = project_root / "experiments" / "grounding" / run_id
run_dir.mkdir(parents=True, exist_ok=False)

manifest = {
    "run_id": run_id,
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "adapter": None,
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "packages": {p: metadata.version(p) for p in [
        "torch", "transformers", "accelerate", "bitsandbytes"
    ]},
    "benchmark_sha256": hashlib.sha256(benchmark_path.read_bytes()).hexdigest(),
    "benchmark": benchmark,
    "messages_by_case": prompts,
    "chat_template": tokenizer.chat_template,
    "tokenizer_class": type(tokenizer).__name__,
    "quantization": quantization.to_dict(),
    "generation_config": model.generation_config.to_dict(),
    "generation_overrides": {
        "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False,
        "use_cache": True, "pad_token_id": tokenizer.pad_token_id,
    },
    "attention_implementation": "eager",
    "nonquantized_dtype": "float16",
}
(run_dir / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Run folder:", run_dir.relative_to(project_root))


Run folder: experiments\grounding\base_20261024T160332_746897Z


In [5]:
results_path = run_dir / "responses.jsonl"
saved_manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
assert saved_manifest["messages_by_case"] == prompts, "Prompt changed; start a new run."
assert saved_manifest["generation_overrides"]["max_new_tokens"] == MAX_NEW_TOKENS
assert saved_manifest["chat_template"] == tokenizer.chat_template

done = set()
if results_path.exists():
    for line in results_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            done.add(json.loads(line)["id"])

with results_path.open("a", encoding="utf-8") as output_file:
    for index, case in enumerate(cases, start=1):
        if case["id"] in done:
            print(f"[{index}/{len(cases)}] {case['id']}: already saved")
            continue
        generated = generate_case(prompts[case["id"]])
        record = {
            **case,
            "model_label": "base",
            **generated,
            "review": {
                "correctness_0_to_2": None,
                "unsupported_claims_count": None,
                "unnecessary_apology": None,
                "unnecessarily_verbose": None,
                "notes": "",
            },
        }
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        output_file.flush()
        print(f"[{index}/{len(cases)}] {case['id']} / {case['prompt_type']} "
              f"— {generated['generation_seconds']:.1f}s "
              f"— token limit: {generated['hit_token_limit']}")
        print(generated["answer"], "\n")

print("Responses saved:", results_path.relative_to(project_root))


[1/24] G01 / direct_fact — 13.6s — token limit: False
The Harbour store closes on Saturday at 17:30. 

[2/24] G02 / direct_fact — 2.5s — token limit: False
Harbour store closes on Saturday at 19:00. 

[3/24] G03 / direct_fact — 2.4s — token limit: False
The Harbour store closes on Saturday at 16:15. 

[4/24] G04 / successful_lookup — 4.4s — token limit: False
The status of ORD-731 is "shipped" and the estimated delivery date is November 12, 2026. 

[5/24] G05 / successful_lookup — 4.2s — token limit: False
The status of ORD-731 is "processing" and the estimated delivery date is November 15, 2026. 

[6/24] G06 / successful_lookup — 4.4s — token limit: False
The status of ORD-731 is "out_for_delivery" and the estimated delivery date is November 9, 2026. 

[7/24] G07 / partial_information — 3.7s — token limit: False
The status of ORD-842 is "processing". The estimated delivery date is not available. 

[8/24] G08 / partial_information — 5.1s — token limit: False
Unfortunately, I couldn't f

## Review results before training v2
The counts below describe execution, not correctness. Review answers semantically against `required_behaviour` and `forbidden_claims`; wording need not match the reference.

Correctness: **0** wrong, **1** partially correct, **2** fully correct. Count unsupported claims separately. A case passes when correctness is 2, unsupported claims are 0, and the answer is not truncated. Record style separately. Report results by prompt type and require all members of a changed-context group to pass for group-level success.

Upload `responses.jsonl` for review. Keep `manifest.json` with it for the later v2 comparison. Do not feed reference answers or scoring metadata to the model.


In [6]:
from collections import Counter

results = [json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()
           if line.strip()]
print(f"Completed: {len(results)}/{len(cases)}")
print("By prompt type:", dict(Counter(r["prompt_type"] for r in results)))
print("Hit token limit:", sum(r["hit_token_limit"] for r in results))
print("Total generation time (seconds):",
      round(sum(r["generation_seconds"] for r in results), 1))


Completed: 24/24
By prompt type: {'direct_fact': 3, 'successful_lookup': 3, 'partial_information': 3, 'missing_identifier': 3, 'unavailable_fact': 3, 'lookup_failure': 3, 'scope_and_conflict': 3, 'action_confirmation': 3}
Hit token limit: 0
Total generation time (seconds): 113.5
